# Project FORESIGHT: Exploratory Data Analysis & Baseline

This notebook covers the initial exploration of the datasets, profiling data quality issues, cleaning the data, and building the baseline model.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Datasets

In [ ]:
sku_master = pd.read_csv("../data/sku_master.csv")
calendar = pd.read_csv("../data/calendar.csv")
sales = pd.read_csv("../data/sales_daily.csv")
inventory = pd.read_csv("../data/inventory_snapshots.csv")

print("SKU Master shape:", sku_master.shape)
print("Calendar shape:", calendar.shape)
print("Sales shape:", sales.shape)
print("Inventory shape:", inventory.shape)

## 2. Profile Data Quality Issues

In [ ]:
# Check for missing values
print("Sales missing values:\n", sales.isnull().sum())
print("\nSKU Master unique categories:\n", sku_master["category"].unique())

# Check for duplicates
print("\nSales duplicates:", sales.duplicated().sum())

## 3. Data Cleaning Pipeline

In [ ]:
def clean_data(sales_df, sku_df):
    # A. Clean SKU categories
    sku_clean = sku_df.copy()
    sku_clean["category"] = sku_clean["category"].str.strip().str.title()
    
    # B. Deduplicate sales
    sales_clean = sales_df.drop_duplicates()
    
    # C. Fill missing units/revenue with median/0 or drop
    sales_clean = sales_clean.dropna(subset=["units_sold", "revenue"])
    
    return sales_clean, sku_clean

sales_clean, sku_clean = clean_data(sales, sku_master)
print("Cleaned sales shape:", sales_clean.shape)
print("Cleaned categories:", sku_clean["category"].unique())

## 4. Exploratory Data Analysis

In [ ]:
# Aggregated weekly sales
sales_clean["date"] = pd.to_datetime(sales_clean["date"])
weekly_sales = sales_clean.resample("W", on="date")["units_sold"].sum()

plt.figure()
plt.plot(weekly_sales, marker='o')
plt.title("Weekly Units Sold Trend")
plt.ylabel("Units")
plt.show()

## 5. Build Seasonal-Naive Baseline

In [ ]:
# Define forecast horizon (e.g. 6 weeks)
# Seasonal-naive baseline predicts demand equal to same period last season
print("Baseline model structure initialized.")